# Amazon Product Reviews — Data Preparation
**Goal**: Download the dataset, explore it, sample 500K rows,
perform a temporal train/val/test split, and save the 3 files to disk.

This notebook is the **single entry point** for all Amazon experiments.
All downstream notebooks (baseline, embeddings, optimisation) read
directly from the saved CSVs — no re-downloading needed.

**Output files**
```
amazon/
  data/
    train.csv    (~80% of 500K)
    val.csv      (~10% of 500K)
    test.csv     (~10% of 500K)
```


## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Download dataset via kagglehub
STEP 3 — Load & inspect raw data
STEP 4 — Understand the columns
STEP 5 — Clean & filter
STEP 6 — Sample 500K rows
STEP 7 — Temporal train / val / test split
STEP 8 — Save to disk
STEP 9 — Sanity checks
```


---
## Step 1 — Imports & Paths

In [ ]:
import os
import json
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Working directory for all Amazon experiments
BASE_DIR = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\amazon"
DATA_DIR = os.path.join(BASE_DIR, "data")

os.makedirs(DATA_DIR, exist_ok=True)
print(f"Base directory : {BASE_DIR}")
print(f"Data directory : {DATA_DIR}")


---
## Step 2 — Download Dataset

We use kagglehub to download the Amazon Product Reviews dataset.
Dataset: saurav9786/amazon-product-reviews
This is the Electronics subset from the UCSD Amazon review corpus.

Run this cell once — kagglehub caches the download locally.
After the first run you can comment it out.


In [ ]:
import kagglehub

# Downloads and caches the dataset — only re-downloads if the version changes
path = kagglehub.dataset_download("saurav9786/amazon-product-reviews")
print(f"Dataset downloaded to: {path}")

# List the files in the downloaded folder
for f in os.listdir(path):
    size_mb = os.path.getsize(os.path.join(path, f)) / 1e6
    print(f"  {f:<50}  {size_mb:.1f} MB")


---
## Step 3 — Load & Inspect Raw Data

The dataset is a CSV with one row per review.
We load it and do a quick inspection before any cleaning.


In [ ]:
# Load the raw file — update filename if kagglehub names it differently
import glob

# Find the CSV file automatically (handles different filenames)
csv_files = glob.glob(os.path.join(path, "*.csv"))
print(f"CSV files found: {csv_files}")

RAW_FILE = csv_files[0]  # take the first one
print(f"Loading: {RAW_FILE}")


In [ ]:
df_raw = pd.read_csv(RAW_FILE)

print(f"Shape    : {df_raw.shape}")
print(f"Columns  : {list(df_raw.columns)}")
print(f"\nFirst 3 rows:")
df_raw.head(3)


In [ ]:
# Check data types and missing values
print("Data types:")
print(df_raw.dtypes)
print(f"\nMissing values:")
print(df_raw.isna().sum())


In [ ]:
# Rating distribution — important to understand what we're predicting
print("Rating distribution:")
print(df_raw['ratings'].value_counts().sort_index())

df_raw['ratings'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.xlabel('Rating'); plt.ylabel('Count')
plt.title('Rating Distribution — Raw Data')
plt.tight_layout(); plt.show()


---
## Step 4 — Understand the Columns

Based on the UCSD Amazon review corpus schema:

| Column | Description | Role |
|--------|-------------|------|
| `userId` | Anonymised reviewer ID | Key column (ignored as feature) |
| `productId` | ASIN — Amazon Standard Identification Number | Key column (ignored as feature) |
| `ratings` | Star rating 1–5 | **Target variable** |
| `timestamp` | Unix timestamp of the review | Used for temporal split |

> **Note**: This dataset is ratings-only (no review text, no product metadata).
> Unlike MovieLens we won't have external signals like IMDB/TMDB.
> The feature engineering will focus on user-level and product-level
> aggregate statistics derived from the ratings themselves.


In [ ]:
# Print a sample to confirm column names
print("Sample rows:")
print(df_raw.head(5).to_string())
print(f"\nTotal reviews : {len(df_raw):,}")
print(f"Unique users  : {df_raw['userId'].nunique():,}")
print(f"Unique products: {df_raw['productId'].nunique():,}")
print(f"Rating range  : {df_raw['ratings'].min()} – {df_raw['ratings'].max()}")


---
## Step 5 — Clean & Filter

**Cleaning steps:**
1. Drop rows with missing ratings (target variable — can't impute)
2. Drop rows with missing userId or productId (key columns — can't identify the interaction)
3. Keep only valid ratings (1–5 integer or half-star values)
4. Parse timestamp into a proper datetime
5. Drop duplicate (userId, productId) pairs — keep the most recent review

**Why drop duplicates?**
A user rating the same product twice creates data leakage — the same
interaction could appear in both train and test. We keep the latest review.


In [ ]:
df = df_raw.copy()

# Step 1: Drop missing targets and keys
before = len(df)
df = df.dropna(subset=['ratings', 'userId', 'productId'])
print(f"Dropped {before - len(df):,} rows with missing ratings/userId/productId")


In [ ]:
# Step 2: Keep only valid ratings (1.0 to 5.0)
before = len(df)
df = df[df['ratings'].between(1.0, 5.0)]
print(f"Dropped {before - len(df):,} rows with out-of-range ratings")


In [ ]:
# Step 3: Parse timestamp
# unixReviewTime is seconds since epoch — convert to datetime
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s', errors='coerce')

# Drop rows where timestamp couldn't be parsed
before = len(df)
df = df.dropna(subset=['datetime'])
print(f"Dropped {before - len(df):,} rows with unparseable timestamps")
print(f"Date range: {df['datetime'].min()} → {df['datetime'].max()}")


In [ ]:
# Step 4: Drop duplicate (userId, productId) pairs — keep most recent
before = len(df)
df = df.sort_values('datetime').drop_duplicates(
    subset=['userId', 'productId'], keep='last'
)
print(f"Dropped {before - len(df):,} duplicate (user, product) pairs")
print(f"Clean dataset size: {len(df):,} rows")


---
## Step 6 — Sample 500K Rows

The full dataset may have millions of rows.
We sample 500K for consistency with our experimental setup.

**Why temporal-aware sampling?**
We sample randomly here (before the temporal split) to keep the
time distribution representative. Sampling after splitting could
accidentally exclude rare time periods from val/test.


In [ ]:
SAMPLE_SIZE = 500_000
SEED        = 42

if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    print(f"Sampled {SAMPLE_SIZE:,} rows from {len(df_raw):,} total")
else:
    print(f"Dataset has only {len(df):,} rows — using all of them")

print(f"Working dataset: {len(df):,} rows")
print(f"Date range after sampling: {df['datetime'].min()} → {df['datetime'].max()}")


---
## Step 7 — Temporal Train / Val / Test Split

**Why temporal split?**
Same reasoning as MovieLens — a recommendation model trained on past
behaviour should be evaluated on future behaviour. Random splits would
leak future ratings into training data.

**Split strategy:**
We use percentile-based cutoffs rather than fixed dates.
This ensures consistent split sizes regardless of the dataset's
date range, which can vary between datasets.

- Train : earliest 70% of timestamps
- Val   : next 15%
- Test  : latest 15%


In [ ]:
# Sort by datetime to find cutoff points
df_sorted = df.sort_values('datetime')

n         = len(df_sorted)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

# Use the datetime at each cutoff as the boundary
train_cutoff = df_sorted.iloc[train_end]['datetime']
val_cutoff   = df_sorted.iloc[val_end]['datetime']

print(f"Train cutoff : {train_cutoff}")
print(f"Val cutoff   : {val_cutoff}")


In [ ]:
# Apply the split
df_train = df[df['datetime'] <  train_cutoff].copy()
df_val   = df[(df['datetime'] >= train_cutoff) & (df['datetime'] < val_cutoff)].copy()
df_test  = df[df['datetime'] >= val_cutoff].copy()

print(f"Train : {len(df_train):,} rows  ({len(df_train)/len(df)*100:.1f}%)")
print(f"Val   : {len(df_val):,} rows   ({len(df_val)/len(df)*100:.1f}%)")
print(f"Test  : {len(df_test):,} rows  ({len(df_test)/len(df)*100:.1f}%)")


In [ ]:
# Verify no overlap between splits
train_idx = set(df_train.index)
val_idx   = set(df_val.index)
test_idx  = set(df_test.index)

assert len(train_idx & val_idx)  == 0, "Overlap between train and val!"
assert len(train_idx & test_idx) == 0, "Overlap between train and test!"
assert len(val_idx   & test_idx) == 0, "Overlap between val and test!"
assert len(train_idx) + len(val_idx) + len(test_idx) == len(df), "Rows don't add up!"

print("No overlaps found — split is clean.")


In [ ]:
# Rating distribution across splits — should be similar
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, (split_name, split_df) in zip(axes, [('Train', df_train),
                                               ('Val',   df_val),
                                               ('Test',  df_test)]):
    split_df['ratings'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'{split_name} ({len(split_df):,} rows)')
    ax.set_xlabel('Rating')
plt.suptitle('Rating Distribution Across Splits', y=1.02)
plt.tight_layout(); plt.show()


---
## Step 8 — Save to Disk

Save the 3 splits as CSV files in the `data/` folder.
All downstream notebooks load directly from these files.


In [ ]:
train_path = os.path.join(DATA_DIR, 'train.csv')
val_path   = os.path.join(DATA_DIR, 'val.csv')
test_path  = os.path.join(DATA_DIR, 'test.csv')

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path,     index=False)
df_test.to_csv(test_path,   index=False)

print("Files saved:")
for fpath in [train_path, val_path, test_path]:
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {fpath}")
    print(f"    {size_mb:.1f} MB")


---
## Step 9 — Sanity Checks

Quick verification that the saved files load correctly
and contain what we expect.


In [ ]:
# Reload and verify
train_check = pd.read_csv(train_path)
val_check   = pd.read_csv(val_path)
test_check  = pd.read_csv(test_path)

print("Reloaded file shapes:")
print(f"  train : {train_check.shape}")
print(f"  val   : {val_check.shape}")
print(f"  test  : {test_check.shape}")


In [ ]:
# Final summary
total = len(df_train) + len(df_val) + len(df_test)
print("=" * 55)
print("AMAZON REVIEWS — DATA PREPARATION COMPLETE")
print("=" * 55)
print(f"  Total rows       : {total:,}")
print(f"  Train            : {len(df_train):,}  ({len(df_train)/total*100:.1f}%)")
print(f"  Val              : {len(df_val):,}   ({len(df_val)/total*100:.1f}%)")
print(f"  Test             : {len(df_test):,}   ({len(df_test)/total*100:.1f}%)")
print(f"  Unique users     : {df['userId'].nunique():,}")
print(f"  Unique products  : {df['productId'].nunique():,}")
print(f"  Target column    : ratings  ({df['ratings'].min()}–{df['ratings'].max()})")
print(f"  Split type       : Temporal (70 / 15 / 15)")
print("=" * 55)
print(f"\nFiles saved to: {DATA_DIR}")
